# EXP-DET-001 — 적대적 입력 탐지 계측 (Colab)

**한이음 HC160 · 얼굴 인증 방어 연구**

---

## 무엇을 하나요

웹캠으로 얼굴을 잠깐 촬영하면서, 이미지를 여러 방식으로 흐리게 만들었을 때
얼굴 특징이 얼마나 변하는지를 **숫자로만** 기록합니다. 이 숫자로 적대적 공격을
탐지하는 기준값을 정합니다.

## 무엇이 기록되고 무엇이 기록되지 않나요

| 기록됨 | 기록되지 않음 |
|---|---|
| 코사인 유사도 등 숫자 | 얼굴 사진·영상 |
| `p02` 같은 익명 라벨 | 이름·이메일·연락처 |
| 카메라 해상도, 처리 속도 | 얼굴 특징 벡터(임베딩) |

- 촬영한 프레임은 **파일로 저장되지 않습니다.** Google Colab 임시 서버의 메모리에서
  처리된 뒤 즉시 사라지고, 세션을 닫으면 서버 자체가 삭제됩니다.
- 다운로드되는 결과 파일에는 숫자와 익명 라벨만 들어 있습니다. 직접 열어서 확인할 수 있습니다.
- 코드가 절대 경로와 생체 원본을 산출물에 넣지 못하도록 막고 있습니다(`probe_log.py`).

## 걸리는 시간

설치 2분 + 촬영 2분 정도입니다. 언제든 런타임을 중단하면 됩니다.


---
## 1단계 · 설치

저장소를 받고 필요한 라이브러리를 설치합니다. 1~2분쯤 걸립니다.

> 설치 로그가 길게 나오는 건 정상입니다. 빨간 경고가 있어도 다음 셀의
> 확인이 통과하면 문제없습니다.


In [ ]:
BRANCH = "experiment/EXP-DET-001-camera-squeeze-probe"

# 저장소를 받는다.
!rm -rf /content/hc160
!git clone --depth 1 --branch $BRANCH https://github.com/Yaho03/26_HC160.git /content/hc160

# facenet-pytorch는 오래된 torch를 요구해서 Colab의 torch를 다운그레이드하려 한다.
# 그 과정에서 설치가 깨지거나 torch가 망가진다. Colab에는 필요한 의존성
# (torch, torchvision, numpy, pillow, requests)이 이미 있으므로 --no-deps로 넣는다.
!pip install --no-deps facenet-pytorch

import sys
sys.path.insert(0, '/content/hc160')


### 설치 확인 ← **꼭 실행하세요**

여기서 실패하면 다음 단계로 넘어가도 계속 에러가 납니다.
`런타임 → 세션 다시 시작` 후 1단계부터 다시 실행해보세요.


In [ ]:
# 설치가 실제로 됐는지 여기서 확인한다. 뒤에서 터지면 원인을 찾기 어렵다.
problems = []

try:
    import torch, torchvision
    print('torch', torch.__version__, '/ torchvision', torchvision.__version__)
except Exception as error:
    problems.append(f'torch 로드 실패: {error}')

try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
    print('facenet-pytorch OK')
except Exception as error:
    problems.append(f'facenet-pytorch 로드 실패: {error}')

try:
    from src.verification.defenses.squeeze_probe import TRANSFORM_ORDER
    print('저장소 모듈 OK  (변환', len(TRANSFORM_ORDER), '종)')
except Exception as error:
    problems.append(f'저장소 모듈 로드 실패: {error}')

print()
if problems:
    for item in problems:
        print('✗', item)
    print()
    print('해결 방법')
    print('  1) 상단 메뉴 → 런타임 → 세션 다시 시작 후 1단계부터 다시 실행')
    print('  2) 그래도 안 되면 위 설치 셀의 출력을 캡처해서 보내주세요')
    raise SystemExit('설치 확인 실패. 다음 단계로 넘어가지 마세요.')

print('✓ 설치 확인 완료. 다음 단계로 진행하세요.')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (CPU로 진행)')


---
## 2단계 · 참가자 라벨 ← **여기만 수정**

`SUBJECT_ID`를 배정받은 익명 라벨로 바꿔주세요. 이름이나 이메일을 넣으면 코드가 거부합니다.


In [ ]:
SUBJECT_ID = "p02"   # ← 배정받은 라벨로 변경

TARGET_FRAMES = 200      # 기록할 프레임 수
ATTACK_EVERY  = 5        # 몇 프레임마다 공격 샘플을 만들지
ATTACK_KINDS  = ["pgd", "fgsm", "pgd_low_eps"]

from src.verification.defenses.probe_log import _require_opaque
_require_opaque('subject_id', SUBJECT_ID)
print('라벨 확인 완료:', SUBJECT_ID)


---
## 3단계 · 카메라 연결

실행하면 브라우저가 카메라 권한을 물어봅니다. 허용해주세요.
화면에 본인 모습이 보이면 연결된 것입니다.

> 프레임은 **PNG 무손실**로 가져옵니다. JPEG로 가져오면 이미 압축된 이미지가 되어
> 측정값이 왜곡됩니다.


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js

display(Javascript('''
window._hc160 = {};
async function hc160_start() {
  const div = document.createElement('div');
  document.body.appendChild(div);
  const video = document.createElement('video');
  video.style.width = '320px';
  video.autoplay = true; video.muted = true; video.playsInline = true;
  div.appendChild(video);
  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 1280, height: 720}});
  video.srcObject = stream;
  await video.play();
  const canvas = document.createElement('canvas');
  window._hc160 = {video, stream, canvas, div};
  const t = video.srcObject.getVideoTracks()[0].getSettings();
  return JSON.stringify({width: t.width, height: t.height, frameRate: t.frameRate});
}
// PNG 무손실. JPEG로 뽑으면 이미 압축된 입력이 되어 측정이 왜곡된다.
async function hc160_grab(n) {
  const {video, canvas} = window._hc160;
  canvas.width = video.videoWidth; canvas.height = video.videoHeight;
  const ctx = canvas.getContext('2d');
  const out = [];
  for (let i = 0; i < n; i++) {
    ctx.drawImage(video, 0, 0);
    out.push(canvas.toDataURL('image/png'));
    await new Promise(r => setTimeout(r, 60));
  }
  return JSON.stringify(out);
}
function hc160_stop() {
  const s = window._hc160;
  if (!s || !s.stream) return 'already stopped';
  s.stream.getTracks().forEach(t => t.stop());
  s.div.remove();
  window._hc160 = {};
  return 'stopped';
}
'''))

import json as _json
camera_info = _json.loads(eval_js('hc160_start()'))
print('카메라 연결됨:', camera_info)


---
## 4단계 · 얼굴 등록

화면을 정면으로 보고 실행하세요. 얼굴이 잡히면 기준으로 등록됩니다.


In [ ]:
import base64, io, json
import numpy as np
from PIL import Image

from src.verification.defenses.verification_defense_temporal_camera import detect_and_crop
from src.verification.defenses.facenet_embed import get_embedding

def grab_frames(n=1):
    """브라우저에서 PNG 프레임 n장을 가져와 BGR ndarray로 돌려준다."""
    import cv2
    payload = json.loads(eval_js(f'hc160_grab({n})'))
    frames = []
    for data_url in payload:
        raw = base64.b64decode(data_url.split(',', 1)[1])
        rgb = np.array(Image.open(io.BytesIO(raw)).convert('RGB'))
        frames.append(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    return frames

enroll_crop = None
for attempt in range(10):
    frame = grab_frames(1)[0]
    crop, _ = detect_and_crop(frame)
    if crop is not None:
        enroll_crop = crop
        break
    print(f'얼굴을 찾지 못했습니다. 다시 시도 {attempt + 1}/10')

assert enroll_crop is not None, '얼굴을 찾지 못했습니다. 조명을 밝게 하고 정면을 봐주세요.'
enroll_torch = get_embedding(enroll_crop)
enroll_vector = enroll_torch.numpy().astype('float64')
print('등록 완료')


---
## 5단계 · 계측

**자연스럽게 움직여주세요.** 고개 각도, 표정, 카메라와의 거리를 조금씩 바꿔주세요.
가만히 응시하면 데이터가 한쪽으로 치우쳐 결과를 쓸 수 없게 됩니다.

2분쯤 걸립니다.


In [ ]:
import secrets, time
from pathlib import Path
from datetime import datetime, timezone

from src.verification.defenses.squeeze_probe import (
    TRANSFORM_ORDER, TRANSFORM_PARAMS, jpeg_headroom, probe_crop,
)
from src.verification.defenses.probe_log import ProbeWriter, write_session_sidecar
from src.verification.defenses.probe_attacks import (
    AttackConfig, attack_for_index, build_attack_params, run_attack,
)
from src.verification.defenses.probe_capture import sample_id, should_attack
from src.verification.defenses.facenet_embed import FaceNetBatchEmbedder

session_id = secrets.token_hex(6)
out_dir = Path('/content/probe_out') / session_id
attack_config = AttackConfig()
attack_params = build_attack_params(ATTACK_KINDS, attack_config)   # 촬영 전 검증

embedder = FaceNetBatchEmbedder()
counters = {'frames_read': 0, 'read_failures': 0, 'frames_without_face': 0,
            'samples_clean': 0, 'samples_adversarial': 0, 'rows': 0}
attack_counts = {k: 0 for k in ATTACK_KINDS}
headroom, attack_index, frame_idx = [], 0, 0
started = time.perf_counter()

with ProbeWriter(out_dir / 'probe.csv', session_id=session_id, subject_id=SUBJECT_ID) as writer:
    while counters['samples_clean'] < TARGET_FRAMES:
        for frame in grab_frames(5):
            if counters['samples_clean'] >= TARGET_FRAMES:
                break
            counters['frames_read'] += 1
            frame_ts_ms = (time.perf_counter() - started) * 1000.0
            crop, _ = detect_and_crop(frame)
            if crop is None:
                counters['frames_without_face'] += 1
                continue

            headroom.append(jpeg_headroom(crop))
            reading = probe_crop(crop, enroll_vector, embedder)
            counters['rows'] += writer.write_sample(
                sample_id=sample_id(frame_idx, 'clean'), frame_idx=frame_idx,
                frame_ts_ms=frame_ts_ms, dropped_frames=counters['read_failures'],
                label='clean', reading=reading)
            counters['samples_clean'] += 1

            if should_attack(frame_idx, ATTACK_EVERY):
                kind = attack_for_index(attack_index, ATTACK_KINDS)
                adv_crop, _ = run_attack(kind, crop, enroll_torch, attack_config)
                adv_reading = probe_crop(adv_crop, enroll_vector, embedder)
                counters['rows'] += writer.write_sample(
                    sample_id=sample_id(frame_idx, 'adversarial'), frame_idx=frame_idx,
                    frame_ts_ms=frame_ts_ms, dropped_frames=counters['read_failures'],
                    label='adversarial', reading=adv_reading, attack_kind=kind)
                counters['samples_adversarial'] += 1
                attack_counts[kind] += 1
                attack_index += 1
            frame_idx += 1
        print(f"  {counters['samples_clean']}/{TARGET_FRAMES}", end='\r')

elapsed = time.perf_counter() - started
print(f"\n완료: {counters['rows']}행 (clean {counters['samples_clean']} / adversarial {counters['samples_adversarial']})")


---
## 6단계 · 카메라 끄기 + 결과 저장


In [ ]:
print(eval_js('hc160_stop()'))

sidecar = {
    'session_id': session_id, 'subject_id': SUBJECT_ID,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'git_commit': None, 'source': 'colab_browser_png',
    'camera': {'index': None, 'width': camera_info.get('width'),
               'height': camera_info.get('height'),
               'fps_nominal': camera_info.get('frameRate'), 'fourcc': 'PNG'},
    'model': {'name': 'InceptionResnetV1', 'pretrained': 'vggface2',
              'weights_file': '20180402-114759-vggface2.pt', 'weights_sha256': None,
              'preprocess': 'resize160_bilinear,(x-127.5)/128.0'},
    'transforms': {n: TRANSFORM_PARAMS[n] for n in TRANSFORM_ORDER},
    'attack': {'kinds': ATTACK_KINDS, 'params': attack_params,
               'counts': attack_counts, 'every': ATTACK_EVERY,
               'target': 'enroll_template'},
    'counters': counters, 'target_frames': TARGET_FRAMES,
    'completed': counters['samples_clean'] >= TARGET_FRAMES,
    'interrupted_by': None,
    'elapsed_sec': round(elapsed, 3),
    'effective_fps': round(counters['frames_read'] / elapsed, 2) if elapsed > 0 else None,
    'jpeg_headroom_q75': round(sum(headroom) / len(headroom), 4) if headroom else None,
    'notes': 'Colab 브라우저 PNG 캡처. dropped_frames는 의미 없음.',
}
write_session_sidecar(out_dir / 'session.json', sidecar)

print('jpeg_headroom_q75 =', sidecar['jpeg_headroom_q75'], ' (참고: 로컬 웹캠 2.2~3.2, 압축 이미지 0.1 미만)')

import shutil
archive = shutil.make_archive(f'/content/probe_{SUBJECT_ID}_{session_id}', 'zip', out_dir)
print('저장:', archive)


---
## 6.5단계 · 품질 자가 검사 ← **꼭 실행하세요**

데이터가 쓸 수 있는지 지금 확인합니다. **실패가 나오면 이 자리에서 다시 찍는 게
훨씬 낫습니다.** 나중에 발견하면 다시 부탁드려야 합니다.


In [ ]:
import numpy as np
from src.verification.defenses.probe_analyze import load_probe_rows, feature_table

# 움직임 판정은 절대값이 아니라 시계열 구조로 한다. 특정 세션의 변동계수를 기준으로
# 삼으면 그 세션이 좋았는지 알 수 없는 순환 논리가 된다. 자기상관과 인접 변화량은
# "정지했는가"를 구조적으로 잡아내므로 기준 세션이 필요 없다.
MIN_HEADROOM   = 1.0    # 로컬 2.2~3.2. 압축된 입력이면 0.1 미만
MIN_FACE_RATE  = 0.50   # 얼굴이 잡힌 프레임 비율
MAX_AUTOCORR   = 0.60   # 인접 프레임 상관. 정지 상태면 0.9 이상이 된다
MIN_STEP_RATIO = 0.30   # 인접 변화량 / 전체 표준편차. 정지면 0에 가깝다
MIN_CLEAN      = 100    # 목표 FPR 1%를 관측값으로 맞추려면 최소 100

problems = []

headroom_value = sidecar['jpeg_headroom_q75']
if headroom_value is None or headroom_value < MIN_HEADROOM:
    problems.append(
        f'입력이 이미 압축된 것 같습니다 (headroom {headroom_value}). '
        '카메라 설정이나 브라우저 문제일 수 있습니다.')

face_rate = counters['frames_read'] and (
    (counters['frames_read'] - counters['frames_without_face']) / counters['frames_read'])
if not face_rate or face_rate < MIN_FACE_RATE:
    problems.append(
        f'얼굴이 잡힌 프레임이 {face_rate:.0%}뿐입니다. '
        '조명을 밝게 하고 화면 안에 얼굴이 계속 보이게 해주세요.')

if counters['samples_clean'] < MIN_CLEAN:
    problems.append(f'표본이 {counters["samples_clean"]}개로 부족합니다. 최소 {MIN_CLEAN}개 필요합니다.')

rows_loaded = load_probe_rows(out_dir / 'probe.csv')
table = feature_table(rows_loaded)
key = ('jpeg_q75', 'self_consistency')
if key in table and len(table[key]['clean']) > 20:
    values = np.array(table[key]['clean'])
    autocorr = float(np.corrcoef(values[:-1], values[1:])[0, 1])
    step_ratio = float(np.median(np.abs(np.diff(values))) / values.std())
    print(f'움직임 지표: 자기상관 {autocorr:+.3f} (기준 {MAX_AUTOCORR} 미만), '
          f'인접 변화량 {step_ratio:.3f} (기준 {MIN_STEP_RATIO} 이상)')
    if autocorr > MAX_AUTOCORR or step_ratio < MIN_STEP_RATIO:
        problems.append(
            f'움직임이 너무 적습니다 (자기상관 {autocorr:+.3f}, 변화량 {step_ratio:.3f}). '
            '고개 각도, 표정, 카메라와의 거리를 더 바꿔가며 다시 찍어주세요.')

print(f'headroom {headroom_value}   얼굴 검출률 {face_rate:.0%}   clean {counters["samples_clean"]}개')
print()
if problems:
    print('다시 촬영이 필요합니다:')
    for item in problems:
        print(' -', item)
    print()
    print('5단계 셀부터 다시 실행해주세요. (3단계 카메라 연결도 다시 필요할 수 있습니다)')
else:
    print('검사 통과. 다음 단계로 진행해주세요.')


---
## 7단계 · 결과 확인 후 전달

아래 셀을 실행하면 파일이 다운로드됩니다. **내용을 직접 열어서 확인해보세요.**
숫자와 익명 라벨만 있고 이미지는 없습니다.


In [ ]:
import csv
with (out_dir / 'probe.csv').open() as f:
    rows = list(csv.reader(f))
print('컬럼:', rows[0])
print('예시 행:', rows[1])
print(f'총 {len(rows) - 1}행')

from google.colab import files
files.download(archive)
